# Assignment 3: From Llama to Mixtral
## Implementing Sparse Mixture of Experts (MoE)
CPSC5910/4910 Large Language Models

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

# Set seed for reproducibility
torch.manual_seed(42)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Running on device: {device}")

# --- Configuration ---
Config = {
    'd_model': 128,              # Hidden dimension size
    'n_heads': 4,                # (For attention, kept simple)
    'seq_len': 16,               # Sequence length per batch
    'batch_size': 2,
    # --- MoE Specific Config ---
    'num_experts': 8,            # Total number of experts available
    'num_experts_per_token': 2,  # Top-K experts selected per token
    'expert_hidden_dim': 256     # Hidden dim inside each expert FFN
}

# --- The Standard SwiGLU Expert (Component used in class) ---
# This will serve as a single "Expert" network.
class SwiGLUExpert(nn.Module):
    def __init__(self, d_model, hidden_dim):
        super().__init__()
        # Projections: d_model -> hidden_dim
        self.w_gate = nn.Linear(d_model, hidden_dim, bias=False)
        self.w_in   = nn.Linear(d_model, hidden_dim, bias=False)
        # Projection back: hidden_dim -> d_model
        self.w_out  = nn.Linear(hidden_dim, d_model, bias=False)

    def forward(self, x):
        # Swish Gating Mechanism: (Swish(xW_gate) * (xW_in))W_out
        return self.w_out(F.silu(self.w_gate(x)) * self.w_in(x))

Running on device: cpu


## Part A: The Gating Mechanism (The Router) [30 Points]

In [2]:
class TopKRouter(nn.Module):
    def __init__(self, d_model, num_experts, top_k):
        super().__init__()
        # The gating linear layer
        self.gate = nn.Linear(d_model, num_experts, bias=False)
        self.top_k = top_k

    def forward(self, x):
        """
        Args: x (Tensor): Hidden states of shape (B, T, D)
        Returns:
            routing_weights (Tensor): (B, T, top_k) - Normalized probabilities for selected experts.
            selected_indices (Tensor): (B, T, top_k) - Indices from 0 to (num_experts-1).
        """
        # 1. Compute raw gating logits: (B, T, num_experts)
        gate_logits = self.gate(x)

        # 2. Select Top-K
        # torch.topk returns (values, indices) along the last dimension.
        # top_k_values shape: (B, T, top_k)
        # selected_indices shape: (B, T, top_k)
        top_k_values, selected_indices = torch.topk(gate_logits, self.top_k, dim=-1)

        # 3. Normalize the Top-K values to get probabilities (weights)
        # Apply softmax along the top_k dimension (-1) so weights sum to 1
        # over the selected experts only.
        routing_weights = F.softmax(top_k_values, dim=-1)

        return routing_weights, selected_indices

# === Test Part A ===
print("\nTesting Router...")
dummy_inputs = torch.randn(Config['batch_size'], Config['seq_len'], Config['d_model']).to(device)
router = TopKRouter(Config['d_model'], Config['num_experts'], Config['num_experts_per_token']).to(device)
weights, indices = router(dummy_inputs)

print(f"Weights shape: {weights.shape} (Expected: B, T, top_k)")
print(f"Indices shape: {indices.shape} (Expected: B, T, top_k)")
# Ensure weights sum to 1 over the top-k dimension
assert torch.allclose(
    weights.sum(dim=-1),
    torch.ones(Config['batch_size'], Config['seq_len']).to(device)
), "Error: Routing weights must sum to 1!"
print("Router Test Passed!")


Testing Router...
Weights shape: torch.Size([2, 16, 2]) (Expected: B, T, top_k)
Indices shape: torch.Size([2, 16, 2]) (Expected: B, T, top_k)
Router Test Passed!


## Part B: The Sparse MoE Layer [40 Points]

In [3]:
class SparseMoELayer(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.num_experts = config['num_experts']

        # 1. The Router
        self.router = TopKRouter(
            config['d_model'],
            self.num_experts,
            config['num_experts_per_token']
        )

        # 2. The Experts (using ModuleList so PyTorch registers parameters)
        self.experts = nn.ModuleList([
            SwiGLUExpert(config['d_model'], config['expert_hidden_dim'])
            for _ in range(self.num_experts)
        ])

    def forward(self, x):
        # x shape: (B, T, D)
        B, T, D = x.shape

        # 1. Get routing data
        # routing_weights shape: (B, T, top_k)
        # selected_indices shape: (B, T, top_k)
        routing_weights, selected_indices = self.router(x)

        # Initialize final output tensor with zeros
        final_output = torch.zeros_like(x)

        # 2. Iterate over all experts and route tokens using masking
        for i in range(self.num_experts):
            expert_layer = self.experts[i]

            # a. Create boolean mask for tokens that chose this expert 'i'.
            #    Check if 'i' exists anywhere in the last dim of selected_indices.
            #    mask shape: (B, T)  — True wherever expert i was selected
            mask = (selected_indices == i).any(dim=-1)  # (B, T)

            # b. Find the specific weight assigned to expert 'i' for each token.
            #    We need shape (B, T): weight value where expert i was chosen, else 0.
            #
            #    selected_indices == i  →  (B, T, top_k) bool
            #    Multiply element-wise with routing_weights, then sum over top_k dim
            #    to collapse to (B, T).  Non-selected positions contribute 0.
            expert_weight = (routing_weights * (selected_indices == i).float()).sum(dim=-1)  # (B, T)

            # c. Apply mask to input x so that tokens NOT routed to expert i
            #    are zeroed out before passing through the expert.
            #    mask.unsqueeze(-1) broadcasts over the D dimension.
            masked_x = x * mask.unsqueeze(-1)  # (B, T, D)

            # d. Run the masked input through expert layer.
            expert_out = expert_layer(masked_x)  # (B, T, D)

            # e. Scale each token's expert output by its routing weight,
            #    then accumulate into final_output.
            #    expert_weight.unsqueeze(-1) broadcasts over D.
            final_output = final_output + expert_out * expert_weight.unsqueeze(-1)

        return final_output

# === Test Part B ===
print("\nTesting MoE Layer...")
moe = SparseMoELayer(Config).to(device)
moe_output = moe(dummy_inputs)
assert moe_output.shape == dummy_inputs.shape, f"MoE output shape incorrect."
assert not torch.allclose(moe_output, dummy_inputs), "MoE Layer seems to be identity function!"
print("MoE Layer Test Passed! Shapes align.")


Testing MoE Layer...
MoE Layer Test Passed! Shapes align.


## Part C: The "Mixtral" Block Assembly [10 Points]

In [4]:
class MixtralBlock(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.ln1 = nn.LayerNorm(config['d_model'])
        # Standard Attention (For simplicity, using PyTorch default here)
        self.attn = nn.MultiheadAttention(
            config['d_model'], config['n_heads'], batch_first=True
        )

        self.ln2 = nn.LayerNorm(config['d_model'])

        # Replace standard dense FFN with SparseMoELayer
        self.ffn = SparseMoELayer(config)

    def forward(self, x):
        # Standard pre-norm residual connection for Attention
        attn_out, _ = self.attn(self.ln1(x), self.ln1(x), self.ln1(x))
        x = x + attn_out

        # Standard pre-norm residual connection for MoE FFN
        x = x + self.ffn(self.ln2(x))
        return x

# === Test Part C ===
print("\nTesting Final Mixtral Block...")
block = MixtralBlock(Config).to(device)
output = block(dummy_inputs)
print(f"Block forward pass successful. Output shape: {output.shape}")


Testing Final Mixtral Block...
Block forward pass successful. Output shape: torch.Size([2, 16, 128])


## Part D: Toy Training Loop & Gradient Verification [20 Points]

In [5]:
# ================= PART D: GRADIENT VERIFICATION LOOP =================
print("\nStarting Part D: Toy Training Loop & Gradient Check...")

# 1. Instantiate Model and Optimizer
block = MixtralBlock(Config).to(device)
# We optimize all parameters in the block, including the router
optimizer = optim.AdamW(block.parameters(), lr=3e-4)
loss_fn = nn.MSELoss()  # Simple loss for toy task

# 2. Create Dummy Data
# Task: Predict input shifted by 1 (force model to do *something*)
dummy_input  = torch.randn(Config['batch_size'], Config['seq_len'], Config['d_model']).to(device)
dummy_target = torch.roll(dummy_input, shifts=1, dims=1)

block.train()

print(f"Initial Router Gate Weight Norm: {block.ffn.router.gate.weight.norm():.4f}")

for step in range(1, 6):  # Run 5 steps
    # a. Zero Gradients
    optimizer.zero_grad()

    # b. Forward Pass
    output = block(dummy_input)

    # c. Compute MSE Loss between output and dummy_target
    loss = loss_fn(output, dummy_target)

    # d. Backward Pass
    loss.backward()

    # e. Optimizer Step
    optimizer.step()

    # Verification: Capture gradient norm of the router's gating layer
    # IF your Part B implementation is differentiable, this MUST be > 0.0
    router_grad_norm = block.ffn.router.gate.weight.grad.norm()
    print(f"Step {step}: Loss = {loss.item():.6f} | Router Grad Norm = {router_grad_norm:.6f}")

print("\nEnd of Training Loop.")
final_router_norm = block.ffn.router.gate.weight.norm()
print(f"Final Router Gate Weight Norm: {final_router_norm:.4f}")

# Final assertion to ensure the weights actually moved from initialization
assert final_router_norm > 0.0 and block.ffn.router.gate.weight.grad is not None
print("SUCCESS: Gradients are flowing through the router mechanism!")


Starting Part D: Toy Training Loop & Gradient Check...
Initial Router Gate Weight Norm: 1.6309
Step 1: Loss = 2.089525 | Router Grad Norm = 0.031791
Step 2: Loss = 2.059690 | Router Grad Norm = 0.030281
Step 3: Loss = 2.029687 | Router Grad Norm = 0.028661
Step 4: Loss = 2.000725 | Router Grad Norm = 0.030021
Step 5: Loss = 1.971650 | Router Grad Norm = 0.032049

End of Training Loop.
Final Router Gate Weight Norm: 1.6306
SUCCESS: Gradients are flowing through the router mechanism!
